In [8]:
!pip install -q datasets scikit-learn
from datasets import load_dataset

ds = load_dataset("fancyzhx/amazon_polarity")
train = ds["train"].shuffle(seed=42).select(range(5000))
test = ds["test"].shuffle(seed=42).select(range(1000))

print(train)
for i in range(3):
    print(train[i]["label"], "|", train[i]["title"], "|", train[i]["content"][:150])

README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  260MB            

amazon_polarity/train-00000-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/train-00001-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  258MB            

amazon_polarity/train-00001-of-00004.par(…): downloading bytes:           |  0.00B            

...

amazon_polarity/test-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  117MB            

amazon_polarity/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 5000
})
0 | Anyone who likes this better than the Pekinpah is a moron. | All the pretty people in this film. Even the Rudy character played by Michael Madsen. This is adapted from a Jim Thompson novel for cryin' out loud! T
0 | Author seems mentally unstable | I know that Tom Robbins has a loyal following and I started the book with high expectations. However, I did not enjoy this book as it was too much wor
1 | Spaetzle Noodles | This type of spaetzle maker is easier to manuveur than the old press kind and much easier on the hands. The difference is that this new spaetzle maker


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train = vec.fit_transform(train["content"])
X_test = vec.transform(test["content"])

clf = LogisticRegression(max_iter=1000).fit(X_train, train["label"])
pred = clf.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(test["label"], pred), 3))
print("Baseline F1:", round(f1_score(test["label"], pred), 3))

Baseline accuracy: 0.825
Baseline F1: 0.829


In [10]:
def baseline_predict(text):
    return "positive" if clf.predict(vec.transform([text]))[0] == 1 else "negative"

print(baseline_predict("Great product, works perfectly and arrived early!"))
print(baseline_predict("Stopped working after two days. Waste of money."))
print(baseline_predict("The battery life is amazing but the camera is disappointing."))

positive
negative
positive


In [11]:
import numpy as np
wrong = np.where(pred != np.array(test["label"]))[0][:8]
for i in wrong:
    print("TRUE:", test[int(i)]["label"], "PRED:", pred[i], "|", test[int(i)]["content"][:200], "\n")

TRUE: 1 PRED: 0 | The product works fine. I ordered the more exprensive one after I read reviews from others on Amazon. My husband likes the presser. It does a good job pressing his pants. However, it was damaged in th 

TRUE: 1 PRED: 0 | The other plastic garages and service stations that we have purchased in the past were made so poorly, but this one has really stood the test of time with two kids under the age of five. 

TRUE: 1 PRED: 0 | Nirvana only got popular after cobain killed himself? Obviously you're an idiot. Nirvana got mega-popular after "Smells like teen spirit" was debuted on MTV, and remained popular up until he killed hi 

TRUE: 0 PRED: 1 | Let Me Call You Sweetheart, by Mary Higgins Clark, read by Bess Armstrong. A convoluted and seemingly implausible tale. Not MHC's best by a long shot, but may have lost a lot in the abridgement, which 

TRUE: 1 PRED: 0 | I bought this dvd for my wife because she wanted a short, effective workout. Right at the bat, Jillian set the b

In [12]:
import torch
print("GPU available:", torch.cuda.is_available())

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch["content"], truncation=True, max_length=128)

train_tok = train.map(tokenize, batched=True)
test_tok = test.map(tokenize, batched=True)
print("Tokenized:", len(train_tok), "train /", len(test_tok), "test")

GPU available: True


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized: 5000 train / 1000 test


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

args = TrainingArguments(
    output_dir="bert-sentiment",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.265924,0.894000,0.893788
2,0.289914,0.291969,0.901000,0.903977


TrainOutput(global_step=626, training_loss=0.2714742959116975, metrics={'train_runtime': 37.8923, 'train_samples_per_second': 263.906, 'train_steps_per_second': 16.521, 'total_flos': 331040168847552.0, 'train_loss': 0.2714742959116975, 'epoch': 2.0})

In [14]:
result = trainer.evaluate()
print("BERT accuracy:", round(result["eval_accuracy"], 3))
print("BERT F1:", round(result["eval_f1"], 3))

preds_bert = trainer.predict(test_tok).predictions.argmax(-1)
true = np.array(test["label"])
print("BERT correct on baseline's first 8 mistakes:", int((preds_bert[wrong] == true[wrong]).sum()), "/ 8")

def bert_predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return "positive" if logits.argmax(-1).item() == 1 else "negative"

for t in ["Great product, works perfectly and arrived early!",
          "Stopped working after two days. Waste of money.",
          "The battery life is amazing but the camera is disappointing."]:
    print(bert_predict(t), "|", t)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.289914,0.291969,2,0.901000,0.903977


BERT accuracy: 0.901
BERT F1: 0.904


BERT correct on baseline's first 8 mistakes: 4 / 8
positive | Great product, works perfectly and arrived early!
negative | Stopped working after two days. Waste of money.
negative | The battery life is amazing but the camera is disappointing.


In [15]:
bert_wrong = np.where(preds_bert != true)[0][:8]
for i in bert_wrong:
    print("TRUE:", true[i], "PRED:", preds_bert[i], "|", test[int(i)]["content"][:200], "\n")

TRUE: 1 PRED: 0 | The other plastic garages and service stations that we have purchased in the past were made so poorly, but this one has really stood the test of time with two kids under the age of five. 

TRUE: 1 PRED: 0 | Nirvana only got popular after cobain killed himself? Obviously you're an idiot. Nirvana got mega-popular after "Smells like teen spirit" was debuted on MTV, and remained popular up until he killed hi 

TRUE: 0 PRED: 1 | What matters is using the cable with the highest capability your gear will support (HDMI, DVI, component), and using a decent quality shielded cable. But don't waste your money on "features" that won' 

TRUE: 1 PRED: 0 | Just a heads up. Two of the songs on 1000 Fires are Babble tunes. If you are a Babble fan, then you probably want to check this out.... 

TRUE: 1 PRED: 0 | I have been using a terrible hole puncher at school for the past ten years. (hole guide shifting after every punch, a deluge of holes raining upon the classroom floor, fractured

In [16]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy
from collections import Counter
nlp = spacy.load("en_core_web_sm")

reviews = [
    "The battery life on this Samsung phone is amazing, but the camera is disappointing.",
    "Apple support in Hyderabad replaced my charger quickly. Great service!",
    "The screen cracked after a week and the delivery from Amazon was late.",
]

counts = Counter()
for r in reviews:
    doc = nlp(r)
    entities = [(e.text, e.label_) for e in doc.ents]
    features = [c.root.text.lower() for c in doc.noun_chunks if c.root.pos_ == "NOUN"]
    counts.update(features)
    print(bert_predict(r), "|", entities, "|", features)

print("Most mentioned features:", counts.most_common(5))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 137.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
negative | [('Samsung', 'ORG')] | ['life', 'phone', 'camera']
positive | [('Apple', 'ORG'), ('Hyderabad', 'ORG')] | ['support', 'charger', 'service']
negative | [('a week', 'DATE'), ('Amazon', 'ORG')] | ['screen', 'week', 'delivery']
Most mentioned features: [('life', 1), ('phone', 1), ('camera', 1), ('support', 1), ('charger', 1)]
